In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10

print("="*80)
print("İNSAN KAYNAKLARI YETENEK ANALİZİ")
print("="*80)

def generate_frequent_itemsets(df_binary, min_support=0.05):
    """
    Binary DataFrame'den frequent itemsets oluşturur
    """
    n_transactions = len(df_binary)
    frequent_itemsets = []
    
    for col in df_binary.columns:
        support = df_binary[col].sum() / n_transactions
        if support >= min_support:
            frequent_itemsets.append({
                'itemset': frozenset([col]),
                'support': support
            })
    
    cols = df_binary.columns.tolist()
    for item1, item2 in combinations(cols, 2):
        support = (df_binary[item1] & df_binary[item2]).sum() / n_transactions
        if support >= min_support:
            frequent_itemsets.append({
                'itemset': frozenset([item1, item2]),
                'support': support
            })
    
    for item1, item2, item3 in combinations(cols, 3):
        support = (df_binary[item1] & df_binary[item2] & df_binary[item3]).sum() / n_transactions
        if support >= min_support:
            frequent_itemsets.append({
                'itemset': frozenset([item1, item2, item3]),
                'support': support
            })
    
    for item1, item2, item3, item4 in combinations(cols, 4):
        support = (df_binary[item1] & df_binary[item2] & df_binary[item3] & df_binary[item4]).sum() / n_transactions
        if support >= min_support:
            frequent_itemsets.append({
                'itemset': frozenset([item1, item2, item3, item4]),
                'support': support
            })
    
    return pd.DataFrame(frequent_itemsets)

def generate_association_rules(frequent_itemsets_df, min_confidence=0.3):
    """
    Frequent itemsets'ten association rules oluşturur
    """
    rules = []
    
    for _, row in frequent_itemsets_df.iterrows():
        itemset = row['itemset']
        if len(itemset) < 2:
            continue
        
        for consequent_item in itemset:
            antecedent = itemset - {consequent_item}
            
            antecedent_row = frequent_itemsets_df[
                frequent_itemsets_df['itemset'] == antecedent
            ]
            
            if len(antecedent_row) > 0:
                antecedent_support = antecedent_row.iloc[0]['support']
                confidence = row['support'] / antecedent_support
                
                if confidence >= min_confidence:
                    consequent_row = frequent_itemsets_df[
                        frequent_itemsets_df['itemset'] == frozenset([consequent_item])
                    ]
                    if len(consequent_row) > 0:
                        consequent_support = consequent_row.iloc[0]['support']
                        lift = confidence / consequent_support
                        
                        rules.append({
                            'antecedents': antecedent,
                            'consequents': frozenset([consequent_item]),
                            'support': row['support'],
                            'confidence': confidence,
                            'lift': lift
                        })
    
    return pd.DataFrame(rules)

print("\n ADIM 1: Veri Yükleme")
print("-" * 80)

df = pd.read_excel(r"C:\Users\seyma\Desktop\web_minning\wm_final_odevi\analize_hazir_is_ilanlari.xlsx")
print(f"✓ {len(df):,} iş ilanı yüklendi")
print(f"✓ {len(df.columns)} kolon mevcut")

beceri_kolonlari = ['Python', 'SQL', 'Excel', 'İngilizce', 'İletişim', 
                    'Liderlik', 'Analiz', 'Takım_Calısması', 'Agile']

teknik_beceriler = ['Python', 'SQL', 'Excel', 'Agile']
sosyal_beceriler = ['İletişim', 'Liderlik', 'Takım_Calısması']
karma_beceriler = ['Analiz', 'İngilizce']  

print(f"\n Beceri Kategorileri:")
print(f"   • Teknik Beceriler: {', '.join(teknik_beceriler)}")
print(f"   • Sosyal Beceriler: {', '.join(sosyal_beceriler)}")
print(f"   • Karma Beceriler: {', '.join(karma_beceriler)}")

print("\n" + "="*80)
print("ADIM 2: Genel İstatistikler ve Beceri Dağılımı")
print("="*80)

beceri_frekanslari = df[beceri_kolonlari].sum().sort_values(ascending=False)
beceri_yuzdeler = (beceri_frekanslari / len(df) * 100).round(2)

print("\n Beceri Talep Sıralaması:")
print("-" * 80)
for beceri, sayi in beceri_frekanslari.items():
    yuzde = beceri_yuzdeler[beceri]
    kategori = "TKN" if beceri in teknik_beceriler else "SOS" if beceri in sosyal_beceriler else "KRM"
    bar = "█" * int(yuzde / 2)
    print(f"[{kategori}] {beceri:<20} │ {sayi:>6,} ilan │ %{yuzde:>5} │ {bar}")

ort_beceri = df['Beceri_Sayisi'].mean()
print(f"\n İlan başına ortalama beceri sayısı: {ort_beceri:.2f}")

print("\n" + "="*80)
print("ADIM 3: TEKNİK vs SOSYAL BECERİ ANALİZİ")
print("="*80)

df['Teknik_Beceri_Sayisi'] = df[teknik_beceriler].sum(axis=1)
df['Sosyal_Beceri_Sayisi'] = df[sosyal_beceriler].sum(axis=1)

toplam_teknik = df['Teknik_Beceri_Sayisi'].sum()
toplam_sosyal = df['Sosyal_Beceri_Sayisi'].sum()

print(f"\n Toplam Beceri Talebi:")
print(f"   • Teknik Beceriler: {toplam_teknik:,} (ilan başına ort: {toplam_teknik/len(df):.2f})")
print(f"   • Sosyal Beceriler: {toplam_sosyal:,} (ilan başına ort: {toplam_sosyal/len(df):.2f})")

teknik_yuzde = toplam_teknik / (toplam_teknik + toplam_sosyal) * 100
sosyal_yuzde = toplam_sosyal / (toplam_teknik + toplam_sosyal) * 100

print(f"\n Beceri Dağılım Oranı:")
print(f"   • Teknik: %{teknik_yuzde:.1f}")
print(f"   • Sosyal: %{sosyal_yuzde:.1f}")

print(f"\n EĞİTİM BÜTÇE ÖNERİSİ:")
if teknik_yuzde > 60:
    print(f"   ► Bütçenin %{teknik_yuzde:.0f}'ini TEKNİK becerilere ayırın!")
    print(f"   ► Öncelikli teknik beceriler: Python, SQL, Excel")
elif sosyal_yuzde > 60:
    print(f"   ► Bütçenin %{sosyal_yuzde:.0f}'ini SOSYAL becerilere ayırın!")
    print(f"   ► Öncelikli sosyal beceriler: İletişim, Liderlik, Takım Çalışması")
else:
    print(f"   ► DENGELI bir eğitim programı önerilir")
    print(f"   ► Teknik: %{teknik_yuzde:.0f}, Sosyal: %{sosyal_yuzde:.0f} oranında")

print("\n" + "="*80)
print("ADIM 4: UNICORN (Bulunması İmkansız) PROFİL ANALİZİ")
print("="*80)

beceri_dagilim = df['Beceri_Sayisi'].value_counts().sort_index()

print("\n İlan Başına İstenen Beceri Sayısı Dağılımı:")
print("-" * 80)
for beceri_say, ilan_say in beceri_dagilim.items():
    yuzde = (ilan_say / len(df) * 100)
    bar = "█" * int(yuzde / 2)
    print(f"{beceri_say} Beceri │ {ilan_say:>6,} ilan │ %{yuzde:>5.1f} │ {bar}")

unicorn_esik = 6
unicorn_ilanlar = df[df['Beceri_Sayisi'] >= unicorn_esik]
unicorn_oran = len(unicorn_ilanlar) / len(df) * 100

print(f"\n UNICORN PROFİL TANIMI: {unicorn_esik}+ beceri isteyen ilanlar")
print(f"   • Unicorn ilan sayısı: {len(unicorn_ilanlar):,}")
print(f"   • Unicorn oran: %{unicorn_oran:.2f}")

medyan_beceri = df['Beceri_Sayisi'].median()
mod_beceri = df['Beceri_Sayisi'].mode()[0]

print(f"\n Sektör Standartları:")
print(f"   • Medyan beceri sayısı: {medyan_beceri:.0f}")
print(f"   • En yaygın beceri sayısı (Mod): {mod_beceri}")

print(f"\n UNICORN DEĞERLENDİRMESİ:")
if unicorn_oran > 20:
    print(f"     UYARI: İlanlarınızın %{unicorn_oran:.1f}'i UNICORN kategorisinde!")
    print(f"   ► Çok fazla beceri talep ediyorsunuz")
    print(f"   ► Öneri: Beceri sayısını {medyan_beceri:.0f}-{medyan_beceri+1:.0f} aralığına çekin")
    print(f"   ► Bu sayede aday havuzunuzu %{(unicorn_oran * 3):.0f} artırabilirsiniz")
elif unicorn_oran > 10:
    print(f"    İlanlarınızın %{unicorn_oran:.1f}'i yüksek beceri talep ediyor")
    print(f"   ► Kritik pozisyonlar için makul, ancak dikkatli olun")
else:
    print(f"    İlanlarınız sektör standartlarıyla UYUMLU")
    print(f"   ► Unicorn oran düşük (%{unicorn_oran:.1f})")
    print(f"   ► Aday havuzunuz geniş olacaktır")

if len(unicorn_ilanlar) > 0:
    print(f"\n En Yaygın Unicorn Profil Kombinasyonları (İlk 5):")
    print("-" * 80)
    for idx, (_, row) in enumerate(unicorn_ilanlar.head(5).iterrows()):
        beceriler = [col for col in beceri_kolonlari if row[col] == 1]
        print(f"   {idx+1}. {' + '.join(beceriler)}")

print("\n" + "="*80)
print(" ADIM 5: BİRLİKTELİK KURALLARI ANALİZİ (Apriori)")
print("="*80)

df_beceri = df[beceri_kolonlari].astype(bool)

print(f"\n  Apriori algoritması çalıştırılıyor...")

frequent_itemsets = generate_frequent_itemsets(df_beceri, min_support=0.05)
frequent_itemsets['length'] = frequent_itemsets['itemset'].apply(len)
print(f"✓ {len(frequent_itemsets)} sık görülen beceri kümesi bulundu (min_support=0.05)")

print(f"\n Beceri Kümesi Dağılımı:")
for uzunluk in sorted(frequent_itemsets['length'].unique()):
    count = len(frequent_itemsets[frequent_itemsets['length'] == uzunluk])
    print(f"   • {uzunluk} becerili kümeler: {count} adet")

rules = generate_association_rules(frequent_itemsets, min_confidence=0.3)
strong_pairs = pd.DataFrame()  

if len(rules) > 0:
    rules = rules.sort_values('lift', ascending=False)
    print(f"\n✓ {len(rules)} birliktelik kuralı oluşturuldu")
    
    rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
    
    print("\n" + "="*80)
    print("EN GÜÇLÜ BİRLİKTELİK KURALLARI (Top 15)")
    print("="*80)
    print("\nMetrik Açıklamaları:")
    print("   • Support: Bu kombinasyonun görülme sıklığı")
    print("   • Confidence: A varsa B'nin olma olasılığı")
    print("   • Lift: Rastgele olmayan birlikteliği (>1 ise anlamlı)")
    print("-" * 80)
    
    for idx, (_, row) in enumerate(rules.head(15).iterrows()):
        print(f"\n KURAL #{idx+1}:")
        print(f"   EĞER   : {row['antecedents_str']}")
        print(f"   O ZAMAN: {row['consequents_str']}")
        print(f"   Support   : %{row['support']*100:.1f}")
        print(f"   Confidence: %{row['confidence']*100:.1f}")
        print(f"   Lift      : {row['lift']:.2f} ({'Güçlü pozitif ilişki' if row['lift'] > 1.5 else 'İlişki var' if row['lift'] > 1 else 'Zayıf ilişki'})")
    
    print("\n" + "="*80)
    print(" ADIM 6: AYRILMAZ BÜTÜN HALİNE GELMİŞ YETKİNLİKLER")
    print("="*80)
    
    strong_pairs = rules[
        (rules['confidence'] >= 0.5) & 
        (rules['lift'] >= 1.3) &
        (rules['antecedents'].apply(len) == 1) &
        (rules['consequents'].apply(len) == 1)
    ].sort_values('lift', ascending=False)
    
    if len(strong_pairs) > 0:
        print(f"\n {len(strong_pairs)} adet güçlü ikili birliktelik bulundu")
        print("-" * 80)
        
        for idx, (_, row) in enumerate(strong_pairs.head(10).iterrows()):
            beceri1 = list(row['antecedents'])[0]
            beceri2 = list(row['consequents'])[0]
            
            kat1 = "Teknik" if beceri1 in teknik_beceriler else "Sosyal" if beceri1 in sosyal_beceriler else "Karma"
            kat2 = "Teknik" if beceri2 in teknik_beceriler else "Sosyal" if beceri2 in sosyal_beceriler else "Karma"
            
            print(f"\n {beceri1} [{kat1}] ⟷ {beceri2} [{kat2}]")
            print(f"   └─ Birlikte görülme: %{row['support']*100:.1f}")
            print(f"   └─ {beceri1} varsa {beceri2} olma olasılığı: %{row['confidence']*100:.1f}")
            print(f"   └─ İlişki gücü (Lift): {row['lift']:.2f}x")
            
            if row['confidence'] >= 0.7:
                print(f"   └─  ÇOK GÜÇLÜ BİRLİKTE! Bu ikisi neredeyse her zaman birlikte isteniyor")
    else:
        print("\n  Yüksek confidence'a sahip ikili birliktelik bulunamadı")
        print("   İlanlar daha çeşitli beceri kombinasyonları içeriyor")
    
    print("\n" + "="*80)
    print(" ADIM 7: DETAYLI BECERİ KOMBİNASYON İSTATİSTİKLERİ")
    print("="*80)
    
    pairs_2 = frequent_itemsets[frequent_itemsets['length'] == 2].sort_values('support', ascending=False)
    if len(pairs_2) > 0:
        print(f"\n🔹 EN SIK GÖRÜLEN 2'Lİ BECERİ KOMBİNASYONLARI (Top 10):")
        print("-" * 80)
        for idx, (_, row) in enumerate(pairs_2.head(10).iterrows()):
            beceriler = list(row['itemset'])
            print(f"   {idx+1}. {' + '.join(beceriler)}")
            print(f"      └─ {len(df) * row['support']:.0f} ilanda görüldü (%{row['support']*100:.1f})")
    
    pairs_3 = frequent_itemsets[frequent_itemsets['length'] == 3].sort_values('support', ascending=False)
    if len(pairs_3) > 0:
        print(f"\n🔸 EN SIK GÖRÜLEN 3'LÜ BECERİ KOMBİNASYONLARI (Top 10):")
        print("-" * 80)
        for idx, (_, row) in enumerate(pairs_3.head(10).iterrows()):
            beceriler = list(row['itemset'])
            print(f"   {idx+1}. {' + '.join(beceriler)}")
            print(f"      └─ {len(df) * row['support']:.0f} ilanda görüldü (%{row['support']*100:.1f})")
    
    pairs_4plus = frequent_itemsets[frequent_itemsets['length'] >= 4].sort_values('support', ascending=False)
    if len(pairs_4plus) > 0:
        print(f"\n EN SIK GÖRÜLEN 4+ BECERİ KOMBİNASYONLARI (Top 5):")
        print("-" * 80)
        for idx, (_, row) in enumerate(pairs_4plus.head(5).iterrows()):
            beceriler = list(row['itemset'])
            print(f"   {idx+1}. {' + '.join(beceriler)}")
            print(f"      └─ {len(df) * row['support']:.0f} ilanda görüldü (%{row['support']*100:.1f})")

print("\n" + "="*80)
print(" ADIM 8: Görselleştirmeler Oluşturuluyor...")
print("="*80)

fig = plt.figure(figsize=(20, 24))

ax1 = plt.subplot(4, 2, 1)
beceri_frekanslari.plot(kind='barh', color='steelblue', ax=ax1)
ax1.set_xlabel('İlan Sayısı', fontsize=11)
ax1.set_ylabel('Beceri', fontsize=11)
ax1.set_title('Beceri Talep Sıralaması\n(Her Becerinin Kaç İlanda İstendiği)', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(beceri_frekanslari):
    ax1.text(v + 50, i, f'{v:,}', va='center', fontsize=9)

ax2 = plt.subplot(4, 2, 2)
colors_pie = ['#3498db', '#e74c3c', '#95a5a6']
sizes = [toplam_teknik, toplam_sosyal, df[karma_beceriler].sum().sum()]
labels = [f'Teknik\n{toplam_teknik:,}\n(%{teknik_yuzde:.1f})', 
          f'Sosyal\n{toplam_sosyal:,}\n(%{sosyal_yuzde:.1f})',
          f'Karma\n{int(sizes[2]):,}']
ax2.pie(sizes, labels=labels, colors=colors_pie, startangle=90, textprops={'fontsize': 10})
ax2.set_title('Teknik vs Sosyal Beceri Dağılımı', fontsize=12, fontweight='bold')

ax3 = plt.subplot(4, 2, 3)
beceri_dagilim.plot(kind='bar', color='coral', ax=ax3)
ax3.set_xlabel('İlan Başına Beceri Sayısı', fontsize=11)
ax3.set_ylabel('İlan Sayısı', fontsize=11)
ax3.set_title('İlan Başına İstenen Beceri Sayısı Dağılımı', fontsize=12, fontweight='bold')
ax3.axvline(x=medyan_beceri-0.5, color='green', linestyle='--', label=f'Medyan ({medyan_beceri:.0f})', linewidth=2)
ax3.axvline(x=unicorn_esik-0.5, color='red', linestyle='--', label=f'Unicorn Eşik ({unicorn_esik})', linewidth=2)
ax3.legend()
ax3.grid(axis='y', alpha=0.3)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=0)

ax4 = plt.subplot(4, 2, 4)
unicorn_data = pd.DataFrame({
    'Kategori': ['Normal İlanlar', 'Unicorn İlanlar'],
    'Sayı': [len(df) - len(unicorn_ilanlar), len(unicorn_ilanlar)]
})
colors_uni = ['#2ecc71', '#e67e22']
bars = ax4.bar(unicorn_data['Kategori'], unicorn_data['Sayı'], color=colors_uni)
ax4.set_ylabel('İlan Sayısı', fontsize=11)
ax4.set_title(f'Unicorn Profil Analizi\n(6+ Beceri İsteyen İlanlar: %{unicorn_oran:.1f})', 
              fontsize=12, fontweight='bold')
for i, (bar, v) in enumerate(zip(bars, unicorn_data['Sayı'])):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 100,
             f'{v:,}\n(%{v/len(df)*100:.1f})',
             ha='center', va='bottom', fontweight='bold', fontsize=10)
ax4.grid(axis='y', alpha=0.3)

ax5 = plt.subplot(4, 2, 5)
teknik_detay = df[teknik_beceriler].sum().sort_values(ascending=False)
teknik_detay.plot(kind='barh', color='#3498db', ax=ax5)
ax5.set_xlabel('İlan Sayısı', fontsize=11)
ax5.set_ylabel('Beceri', fontsize=11)
ax5.set_title('Teknik Becerilerin Detaylı Dağılımı', fontsize=12, fontweight='bold')
ax5.grid(axis='x', alpha=0.3)
for i, v in enumerate(teknik_detay):
    ax5.text(v + 50, i, f'{v:,}', va='center', fontsize=9)

ax6 = plt.subplot(4, 2, 6)
sosyal_detay = df[sosyal_beceriler].sum().sort_values(ascending=False)
sosyal_detay.plot(kind='barh', color='#e74c3c', ax=ax6)
ax6.set_xlabel('İlan Sayısı', fontsize=11)
ax6.set_ylabel('Beceri', fontsize=11)
ax6.set_title('Sosyal Becerilerin Detaylı Dağılımı', fontsize=12, fontweight='bold')
ax6.grid(axis='x', alpha=0.3)
for i, v in enumerate(sosyal_detay):
    ax6.text(v + 50, i, f'{v:,}', va='center', fontsize=9)

if len(frequent_itemsets) > 0:
    ax7 = plt.subplot(4, 2, 7)
    support_dist = frequent_itemsets.groupby('length')['support'].mean()
    support_dist.plot(kind='bar', color='purple', ax=ax7)
    ax7.set_xlabel('Beceri Kümesi Boyutu', fontsize=11)
    ax7.set_ylabel('Ortalama Support', fontsize=11)
    ax7.set_title('Beceri Kümesi Boyutuna Göre Ortalama Support', fontsize=12, fontweight='bold')
    ax7.grid(axis='y', alpha=0.3)
    plt.setp(ax7.xaxis.get_majorticklabels(), rotation=0)

if len(rules) > 0:
    ax8 = plt.subplot(4, 2, 8)
    top_rules = rules.head(10).copy()
    top_rules['rule_name'] = top_rules['antecedents_str'] + ' → ' + top_rules['consequents_str']
    
    top_rules['rule_name'] = top_rules['rule_name'].str[:35]
    
    y_pos = range(len(top_rules))
    ax8.barh(y_pos, top_rules['lift'].values, color='teal')
    ax8.set_yticks(y_pos)
    ax8.set_yticklabels(top_rules['rule_name'].values, fontsize=8)
    ax8.set_xlabel('Lift Değeri', fontsize=11)
    ax8.set_title('En Güçlü Birliktelik Kuralları (Lift)', fontsize=12, fontweight='bold')
    ax8.axvline(x=1, color='red', linestyle='--', label='Lift=1 (Bağımsız)', alpha=0.5)
    ax8.legend()
    ax8.grid(axis='x', alpha=0.3)

plt.suptitle('İK YETENEK ANALİZİ - KAPSAMLI RAPOR\nBirliktelik Kuralları Analizi', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('ik_yetenek_analizi_rapor.png', dpi=300, bbox_inches='tight')
print("✓ Görselleştirmeler kaydedildi: ik_yetenek_analizi_rapor.png")

print("\n" + "="*80)
print(" ADIM 9: Özet Rapor Oluşturuluyor...")
print("="*80)

ozet_satir = []
ozet_satir.append("╔═══════════════════════════════════════════════════════════════════════════╗")
ozet_satir.append("║                   İK YETENEK ANALİZİ - ÖZET RAPOR                        ║")
ozet_satir.append("║                  Birliktelik Kuralları (Apriori) Analizi                  ║")
ozet_satir.append("╚═══════════════════════════════════════════════════════════════════════════╝")
ozet_satir.append("")
ozet_satir.append(" GENEL İSTATİSTİKLER")
ozet_satir.append("="*77)
ozet_satir.append(f"• Toplam İlan Sayısı         : {len(df):,}")
ozet_satir.append(f"• Analiz Edilen Beceri Sayısı: {len(beceri_kolonlari)}")
ozet_satir.append(f"• İlan Başına Ort. Beceri    : {ort_beceri:.2f}")
ozet_satir.append(f"• Medyan Beceri Sayısı       : {medyan_beceri:.0f}")
ozet_satir.append("")
ozet_satir.append(" SORU 1: EĞİTİM BÜTÇESİ TEKNİK Mİ SOSYAL BECERİLERE Mİ HARCANMALI?")
ozet_satir.append("="*77)
ozet_satir.append("")
ozet_satir.append(" Beceri Talep Dağılımı:")
ozet_satir.append(f"   • Teknik Beceriler: {toplam_teknik:,} talep (%{teknik_yuzde:.1f})")
ozet_satir.append(f"   • Sosyal Beceriler: {toplam_sosyal:,} talep (%{sosyal_yuzde:.1f})")
ozet_satir.append("")
ozet_satir.append(" ÖNERİ:")

if teknik_yuzde > 60:
    ozet_satir.append(f"    Eğitim bütçenizin %{teknik_yuzde:.0f}'ini TEKNİK BECERİLERE ayırın!")
    ozet_satir.append("")
    ozet_satir.append("   Öncelikli Teknik Beceriler:")
    for i, (beceri, sayi) in enumerate(teknik_detay.head(3).items(), 1):
        ozet_satir.append(f"   {i}. {beceri}: {sayi:,} ilan (%{sayi/len(df)*100:.1f})")
elif sosyal_yuzde > 60:
    ozet_satir.append(f"    Eğitim bütçenizin %{sosyal_yuzde:.0f}'ini SOSYAL BECERİLERE ayırın!")
    ozet_satir.append("")
    ozet_satir.append("   Öncelikli Sosyal Beceriler:")
    for i, (beceri, sayi) in enumerate(sosyal_detay.head(3).items(), 1):
        ozet_satir.append(f"   {i}. {beceri}: {sayi:,} ilan (%{sayi/len(df)*100:.1f})")
else:
    ozet_satir.append(f"    DENGELI bir eğitim programı uygulayın")
    ozet_satir.append(f"   • Teknik becerilere: %{teknik_yuzde:.0f}")
    ozet_satir.append(f"   • Sosyal becerilere: %{sosyal_yuzde:.0f}")

ozet_satir.append("")
ozet_satir.append(" SORU 2: UNICORN PROFİLLER Mİ ARIYORUZ?")
ozet_satir.append("="*77)
ozet_satir.append("")
ozet_satir.append(f" Unicorn Analizi ({unicorn_esik}+ beceri isteyen ilanlar):")
ozet_satir.append(f"   • Unicorn İlan Sayısı: {len(unicorn_ilanlar):,}")
ozet_satir.append(f"   • Toplam İlanlar İçinde Oran: %{unicorn_oran:.2f}")
ozet_satir.append("")
ozet_satir.append(" DEĞERLENDİRME:")

if unicorn_oran > 20:
    ozet_satir.append(f"     UYARI: İlanlarınızın %{unicorn_oran:.1f}'i UNICORN kategorisinde!")
    ozet_satir.append("")
    ozet_satir.append("    Çok fazla beceri talep ediyorsunuz")
    ozet_satir.append("    Aday havuzunuz çok daralıyor")
    ozet_satir.append("")
    ozet_satir.append("    ÖNERİLER:")
    ozet_satir.append(f"   • Beceri sayısını {medyan_beceri:.0f}-{medyan_beceri+1:.0f} aralığına çekin")
    ozet_satir.append(f"   • Bu sayede aday havuzunuzu %{unicorn_oran * 3:.0f} artırabilirsiniz")
elif unicorn_oran > 10:
    ozet_satir.append(f"    İlanlarınızın %{unicorn_oran:.1f}'i yüksek beceri talep ediyor")
    ozet_satir.append("    Kritik pozisyonlar için makul")
else:
    ozet_satir.append("    İlanlarınız SEKTÖR STANDARTLARIYLA UYUMLU!")
    ozet_satir.append(f"    Unicorn oran düşük (%{unicorn_oran:.1f})")
    ozet_satir.append("    Aday havuzunuz geniş olacak")

ozet_satir.append("")
ozet_satir.append(" SORU 3: AYRILMAZ BÜTÜN HALİNE GELMİŞ YETKİNLİKLER")
ozet_satir.append("="*77)
ozet_satir.append("")
ozet_satir.append(" Birliktelik Kuralları Analizi:")
ozet_satir.append(f"   • Bulunan Sık İtemset: {len(frequent_itemsets)}")
ozet_satir.append(f"   • Oluşturulan Kural Sayısı: {len(rules)}")
ozet_satir.append("")

pairs_3 = frequent_itemsets[frequent_itemsets['length'] == 3].sort_values('support', ascending=False)

if len(strong_pairs) > 0:
    ozet_satir.append(" EN GÜÇLÜ 5 İKİLİ BİRLİKTELİK:")
    ozet_satir.append("="*77)
    for idx, (_, row) in enumerate(strong_pairs.head(5).iterrows(), 1):
        beceri1 = list(row['antecedents'])[0]
        beceri2 = list(row['consequents'])[0]
        ozet_satir.append("")
        ozet_satir.append(f"{idx}. {beceri1} ⟷ {beceri2}")
        ozet_satir.append(f"   • Birlikte görülme oranı: %{row['support']*100:.1f}")
        ozet_satir.append(f"   • {beceri1} varsa {beceri2} olma olasılığı: %{row['confidence']*100:.1f}")
        ozet_satir.append(f"   • İlişki gücü (Lift): {row['lift']:.2f}x")
        if row['confidence'] >= 0.7:
            ozet_satir.append("    ÇOK GÜÇLÜ BİRLİKTE!")
else:
    ozet_satir.append("   ℹ  Yüksek güçte ikili birliktelik bulunamadı")

if len(pairs_3) > 0:
    ozet_satir.append("")
    ozet_satir.append(" EN SIK 3'LÜ KOMBİNASYONLAR:")
    ozet_satir.append("="*77)
    for idx, (_, row) in enumerate(pairs_3.head(3).iterrows(), 1):
        beceriler = ' + '.join(list(row['itemset']))
        ozet_satir.append(f"   {idx}. {beceriler}")
        ozet_satir.append(f"      ({len(df) * row['support']:.0f} ilanda, %{row['support']*100:.1f})")

ozet_satir.append("")
ozet_satir.append("╔═══════════════════════════════════════════════════════════════════════════╗")
ozet_satir.append("║                           RAPOR SONU                                      ║")
ozet_satir.append(f"║                  Analiz Tarihi: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}                              ║")
ozet_satir.append("╚═══════════════════════════════════════════════════════════════════════════╝")

ozet_rapor = '\n'.join(ozet_satir)

with open('ik_yetenek_analizi_ozet.txt', 'w', encoding='utf-8') as f:
    f.write(ozet_rapor)

print("✓ Özet rapor kaydedildi: ik_yetenek_analizi_ozet.txt")
print(ozet_rapor)

print("\n" + "="*80)
print(" ADIM 10: Detaylı Excel Raporu Oluşturuluyor...")
print("="*80)

with pd.ExcelWriter('ik_yetenek_analizi_detay.xlsx', engine='openpyxl') as writer:
    print("✓ Detaylı Excel raporu oluşturuluyor...")   
    genel_stats = pd.DataFrame({
        'Metrik': [
            'Toplam İlan',
            'Ortalama Beceri/İlan',
            'Medyan Beceri',
            'Toplam Teknik Talep',
            'Toplam Sosyal Talep',
            'Unicorn İlan Sayısı',
            'Unicorn Oranı (%)'
        ],
        'Değer': [
            len(df),
            f'{ort_beceri:.2f}',
            medyan_beceri,
            toplam_teknik,
            toplam_sosyal,
            len(unicorn_ilanlar),
            f'{unicorn_oran:.2f}'
        ]
    })
    genel_stats.to_excel(writer, sheet_name='Genel İstatistikler', index=False)
    
    beceri_df = pd.DataFrame({
        'Beceri': beceri_frekanslari.index,
        'İlan Sayısı': beceri_frekanslari.values,
        'Yüzde': beceri_yuzdeler.values
    })
    beceri_df.to_excel(writer, sheet_name='Beceri Frekansları', index=False)
    
    if len(rules) > 0:
        rules_export = rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].copy()
        rules_export.columns = ['Öncül', 'Sonuç', 'Support', 'Confidence', 'Lift']
        rules_export.to_excel(writer, sheet_name='Birliktelik Kuralları', index=False)
    
    if len(frequent_itemsets) > 0:
        freq_export = frequent_itemsets.copy()
        freq_export['itemsets_str'] = freq_export['itemset'].apply(lambda x: ', '.join(list(x)))
        freq_export[['itemsets_str', 'support', 'length']].to_excel(
            writer, sheet_name='Sık İtemsetler', index=False
        )
    
    if len(unicorn_ilanlar) > 0:
        unicorn_export = unicorn_ilanlar[['Baslik'] + beceri_kolonlari + ['Beceri_Sayisi']].head(100).copy()
        unicorn_export.to_excel(writer, sheet_name='Unicorn İlanlar', index=False)

print("✓ Detaylı Excel raporu kaydedildi: ik_yetenek_analizi_detay.xlsx")

print("\n" + "="*80)
print(" ANALİZ TAMAMLANDI!")
print("="*80)
print("\n Oluşturulan Dosyalar:")
print("   1. ik_yetenek_analizi_rapor.png - Görsel raporlar (8 grafik)")
print("   2. ik_yetenek_analizi_ozet.txt - Özet metin raporu")
print("   3. ik_yetenek_analizi_detay.xlsx - Detaylı Excel raporu (5 sayfa)")
print("\n" + "="*80)